# Module 01: NumPy for Machine Learning
## Notebook 02: Indexing, Slicing, Tensor Reshaping, and Einstein Summation

In machine learning workflows, data rarely arrives in the exact shape your models require. You will constantly slice subsets of features, partition samples, reorder image channels, and perform multi-dimensional tensor contractions.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Extract elements, rows, columns, and sub-matrices using multi-dimensional slicing.
2. Distinguish between **memory views** and **independent copies** to avoid data corruption bugs.
3. Reshape arrays using `.reshape()` and understand the `-1` auto-inferred dimension.
4. Add and remove singleton dimensions using `np.newaxis`, `np.expand_dims()`, and `np.squeeze()`.
5. Combine datasets using `np.concatenate()`, `np.vstack()`, and `np.hstack()`.
6. Master **Fancy Indexing memory semantics** (views vs independent allocations).
7. Reorder tensor dimensions for computer vision workflows (`(B, C, H, W)` $\leftrightarrow$ `(B, H, W, C)`).
8. Perform complex tensor contractions using **Einstein Summation (`np.einsum`)**.

In [ ]:
import numpy as np

print(f"NumPy version: {np.__version__}")

### 1. 1D Array Indexing and Slicing

Slicing syntax follows standard Python: `arr[start:stop:step]`
- `start`: Inclusive starting index (defaults to 0).
- `stop`: Exclusive ending index (defaults to length of array).
- `step`: Stride length (defaults to 1; negative step reverses direction).

In [ ]:
arr = np.array([10, 20, 30, 40, 50, 60, 70, 80, 90, 100])

print("Original array:           ", arr)
print("Single element arr[2]:    ", arr[2])
print("Slice arr[2:7]:           ", arr[2:7])
print("Step slice arr[::2]:      ", arr[::2])
print("Reversed array arr[::-1]: ", arr[::-1])
print("Negative indexing arr[-3:]:", arr[-3:])

---
### 2. Multi-Dimensional Indexing (2D & 3D)

In a 2D array representing a dataset:
- **Axis 0** represents **Rows** (samples/observations $N$).
- **Axis 1** represents **Columns** (features/variables $D$).

Syntax: `matrix[row_slice, col_slice]`

In [ ]:
# Simulated dataset: 5 samples, 4 features
# e.g., [Age, Income, Credit Score, Years Employed]
data = np.array([
    [25, 45000, 710, 3],
    [38, 82000, 680, 8],
    [45, 110000, 750, 15],
    [29, 52000, 620, 2],
    [52, 95000, 800, 20]
])

print("Dataset matrix (5 samples x 4 features):")
print(data)

# Extract a single observation (Row 0)
print("\nFirst sample (Sample 0):", data[0, :])

# Extract a single feature across all samples (Feature 1: Income)
income_feature = data[:, 1]
print("Income column for all samples:", income_feature)

# Extract a sub-matrix (First 3 samples, first 2 features)
sub_matrix = data[:3, :2]
print("\nFirst 3 samples, first 2 features:\n", sub_matrix)

---
### 3. Critical Concept: Memory Views vs. Deep Copies

> **CRITICAL WARNING FOR MACHINE LEARNING:**
> In NumPy, standard array slicing produces a **VIEW** of the existing array, **NOT a copy**!
> If you modify a view, you **silently mutate the original data**.
> To create an independent duplicate, you must explicitly call `.copy()`.

In [ ]:
# Demonstrating the 'View' behavior
original_data = np.array([100.0, 200.0, 300.0, 400.0])
view_slice = original_data[:2]

# Modifying the view mutates original_data!
view_slice[0] = 999.0

print("view_slice:    ", view_slice)
print("original_data: ", original_data)  # Original was mutated!
print("Does view share memory?", view_slice.base is original_data)

In [ ]:
# Safe practice: Explicitly creating a .copy()
safe_original = np.array([100.0, 200.0, 300.0, 400.0])
independent_copy = safe_original[:2].copy()

independent_copy[0] = 999.0

print("\nindependent_copy: ", independent_copy)
print("safe_original:    ", safe_original)  # Original remains intact!
print("Does copy share memory?", independent_copy.base is safe_original)

---
### 4. Reshaping Arrays

Reshaping reorganizes data into a new shape without altering the underlying data elements.
- The total number of elements (`arr.size`) must remain identical!
- The special value `-1` allows NumPy to automatically infer the remaining dimension.

#### Flattening Arrays: `flatten()` vs `ravel()`
- `.flatten()`: Returns a **copy** of the flattened 1D array.
- `.ravel()`: Returns a **view** whenever possible (faster, zero memory overhead).

In [ ]:
# Sequence of 12 elements
sequence = np.arange(12)
print("Original 1D array:", sequence)

# Reshape to 3 rows, 4 columns
matrix_3x4 = sequence.reshape(3, 4)
print("\nReshaped to (3, 4):\n", matrix_3x4)

# Using -1 to let NumPy calculate the dimension automatically
matrix_2x6 = sequence.reshape(2, -1)
print("\nReshaped to (2, -1) -> shape is:", matrix_2x6.shape)

# Flattening: ravel (view) vs flatten (copy)
raveled = matrix_3x4.ravel()
flattened = matrix_3x4.flatten()

print("\nRaveled shape:  ", raveled.shape, "| Is view?", raveled.base is not None)
print("Flattened shape:", flattened.shape, "| Is view?", flattened.base is not None)

---
### 5. Adding and Removing Singleton Dimensions

Machine learning libraries (like Scikit-Learn or PyTorch) require specific dimensionalities:
- A 1D target vector `y` of shape `(N,)` often needs to be reshaped to a 2D column matrix `(N, 1)`.
- Tools:
  - `np.newaxis` (or `None`)
  - `np.expand_dims(arr, axis)`
  - `np.squeeze(arr)` removes dimensions of length 1.

In [ ]:
# 1D Target vector
y = np.array([1.2, 3.4, 5.6, 7.8])
print(f"Original y shape: {y.shape} (ndim: {y.ndim})")

# Method 1: Using np.newaxis / None
y_col = y[:, np.newaxis]
print(f"Using np.newaxis shape: {y_col.shape} (ndim: {y_col.ndim})")

# Method 2: Using np.expand_dims
y_expanded = np.expand_dims(y, axis=0) # Shape: (1, 4) - row vector
print(f"Using np.expand_dims(axis=0) shape: {y_expanded.shape}")

# Squeezing out singleton dimensions
squeezed = np.squeeze(y_col)
print(f"After np.squeeze shape: {squeezed.shape}")

---
### 6. Stacking and Concatenation

Combining separate feature sets or appending sample batches:
- `np.concatenate([a, b], axis)`: Joins along an existing axis.
- `np.vstack([a, b])`: Stacks vertically (row-wise, adding samples).
- `np.hstack([a, b])`: Stacks horizontally (column-wise, adding features).
- `np.column_stack([a, b])`: Stacks 1D vectors as columns into a 2D matrix.

In [ ]:
# Feature set 1 (Numerical features: Age, Income)
X_num = np.array([
    [25, 50000],
    [32, 65000],
    [47, 90000]
])

# Feature set 2 (Encoded categorical features: Education, Job)
X_cat = np.array([
    [2, 101],
    [3, 102],
    [1, 101]
])

# Horizontally combine features: shape becomes (3, 4)
X_full = np.hstack([X_num, X_cat])
print("Horizontally stacked feature matrix (3 samples, 4 features):\n", X_full)

# New batch of 2 samples arriving
X_new_batch = np.array([
    [29, 58000, 2, 103],
    [41, 82000, 4, 101]
])

# Vertically combine samples: shape becomes (5, 4)
X_all_samples = np.vstack([X_full, X_new_batch])
print("\nVertically stacked samples (5 samples, 4 features):\n", X_all_samples)

---
### 7. Advanced Usages: Fancy Indexing, Tensor Permutation, and Einstein Summation

#### A. Fancy Indexing (Integer Array Indexing) vs. Slicing Semantics

> **CRITICAL DIFFERENCE:**
> - **Basic Slicing (`arr[0:2]`)** always returns a **VIEW** (shared buffer).
> - **Fancy Indexing (`arr[[0, 2]]` or `arr[row_idx, col_idx]`)** always creates a **NEW INDEPENDENT COPY**!
>
> If you extract elements via fancy indexing and mutate them, the original array is **NEVER** modified.

In [ ]:
matrix = np.array([
    [10, 20, 30],
    [40, 50, 60],
    [70, 80, 90]
])

# Extract specific non-contiguous coordinates: (0, 1) and (2, 2)
row_indices = np.array([0, 2])
col_indices = np.array([1, 2])
selected = matrix[row_indices, col_indices]

print("Extracted elements [0,1] and [2,2]:", selected)
print("Is fancy indexing a view?", selected.base is matrix)

# Modifying 'selected' has zero effect on 'matrix'
selected[0] = 9999
print("Original matrix remains intact:\n", matrix)

#### B. Tensor Permutation & Axis Swapping (Computer Vision Pipelines)

In deep learning:
- **PyTorch format**: Channels-First $\rightarrow$ `(Batch, Channels, Height, Width)` or `(B, C, H, W)`.
- **TensorFlow / OpenCV format**: Channels-Last $\rightarrow$ `(Batch, Height, Width, Channels)` or `(B, H, W, C)`.

Using `np.transpose(tensor, axes)` allows arbitrary axis reordering without data corruption.

In [ ]:
# Batch of 4 RGB images: 32x32 pixels, 3 channels (Channels-Last)
images_tf = np.random.rand(4, 32, 32, 3).astype(np.float32)
print("TensorFlow / OpenCV format shape (B, H, W, C):", images_tf.shape)

# Convert to PyTorch format (B, C, H, W):
# Axis mapping: 0 -> 0 (B), 3 -> 1 (C), 1 -> 2 (H), 2 -> 3 (W)
images_torch = np.transpose(images_tf, (0, 3, 1, 2))
print("Converted to PyTorch format shape (B, C, H, W):  ", images_torch.shape)

# Swap individual axes using np.swapaxes (e.g. swap H and W for a 90-degree transpose)
transposed_hw = np.swapaxes(images_torch, 2, 3)
print("Swapped spatial dimensions (H, W -> W, H):      ", transposed_hw.shape)

#### C. Einstein Summation (`np.einsum`)

`np.einsum` allows you to express virtually any tensor contraction, dot product, outer product, or matrix multiplication using concise subscript strings.

Rules of Einstein Notation:
1. Repeated indices in different input terms mean **multiplication followed by summation** along that axis.
2. Indices omitted from the output string (`->...`) are summed over (contracted).
3. Indices appearing in the output string are retained in that exact axis order.

In [ ]:
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])
u = np.array([1, 2])
v = np.array([3, 4])

# 1. Vector Dot Product: \sum_i u_i * v_i
dot_einsum = np.einsum('i,i->', u, v)
print("1. Vector Dot Product (i,i->):", dot_einsum)

# 2. Outer Product: C_ij = u_i * v_j
outer_einsum = np.einsum('i,j->ij', u, v)
print("2. Outer Product (i,j->ij):\n", outer_einsum)

# 3. Matrix Multiplication: C_ik = \sum_j A_ij * B_jk
matmul_einsum = np.einsum('ij,jk->ik', A, B)
print("3. Matrix Multiplication (ij,jk->ik):\n", matmul_einsum)

# 4. Matrix Trace: \sum_i A_ii
trace_einsum = np.einsum('ii->', A)
print("4. Matrix Trace (ii->):", trace_einsum)

# 5. Batched Matrix Multiplication (Essential for Multi-Head Attention in Transformers!)
# Shape: (Batch=2, M=3, K=4) @ (Batch=2, K=4, N=2) -> (Batch=2, M=3, N=2)
batch_X = np.random.randn(2, 3, 4)
batch_W = np.random.randn(2, 4, 2)
batch_out = np.einsum('bik,bkj->bij', batch_X, batch_W)
print("5. Batched Matrix Multiplication (bik,bkj->bij) Shape:", batch_out.shape)

### Summary & Next Steps
In this notebook, you mastered:
- 1D and multi-dimensional slicing mechanics.
- The vital distinction between memory views and `.copy()`.
- Reshaping arrays with auto-inferred dimensions (`-1`).
- Manipulating singleton dimensions using `np.newaxis` and `np.expand_dims`.
- **Fancy indexing memory semantics** (independent copies vs basic slicing views).
- **Tensor dimension permuting** for computer vision frameworks (`(B, C, H, W)` $\leftrightarrow$ `(B, H, W, C)`).
- **Einstein Summation (`np.einsum`)** for unified, highly optimized tensor contractions.

**Next Notebook:** `03_vectorization_and_broadcasting.ipynb` — Unlock NumPy's vectorization speed, multi-dimensional broadcasting, stable softmax, and one-hot encoding without loops.